# Ordner-Clusterung mit LangGraph

**Bottom-Up Clustering mit Top-Down Grundstruktur.**  
Dateinamen + Metadaten → thematische Cluster → Ordnerstruktur.

Dieses Notebook zeigt einen agentenbasierten Workflow mit **LangGraph**:  
State-Management, Human-in-the-Loop und bedingte Schleifen.

## Architektur-Überblick

```
┌─────────┐     ┌──────────┐     ┌─────────┐
│  START   │────▶│ scannen  │────▶│clustern │
└─────────┘     └──────────┘     └────┬────┘
                                      │
                                      ▼
                              ┌──────────────┐
                         ┌───▶│    review     │◀──┐
                         │    │  (interrupt)  │   │
                         │    └──────┬───────┘   │
                         │           │            │
                         │     ┌─────┴─────┐      │
                         │     ▼           ▼      │
                   ┌───────────┐   ┌──────────────┐
                   │ausfuehren │   │ueberarbeiten │
                   └─────┬─────┘   └──────────────┘
                         │
                         ▼
                    ┌─────────┐
                    │   END   │
                    └─────────┘

  Undo = separate Funktion (außerhalb des Graphen)
```

**Zustände:** `scannen` → `clustern` → `review` ⇄ `ueberarbeiten` → `ausfuehren`  
**Human-in-the-Loop:** Der `review`-Node pausiert via `interrupt()` und wartet auf User-Feedback.

In [ ]:
# === Imports ===
# pip install langgraph langchain-core gradio requests
# pip install python-docx python-pptx openpyxl PyMuPDF Pillow  (für Metadaten)

import os, json, re, shutil, textwrap
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import TypedDict, Annotated
from uuid import uuid4

import requests
import gradio as gr

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

In [ ]:
# === Konfiguration ===

# LM Studio API
API_URL  = "http://localhost:1234/v1/chat/completions"
MODELL   = "qwen/qwen3.6-27b"
MAX_TOKENS  = 100_000
TEMPERATUR  = 0.1

# Skills-Datei (ausgelagerte LLM-Anweisungen)
SKILLS_PFAD = Path("ordner_clustering_skills/clustering_skills.md")

# Sicherheit
UNDO_LOG   = Path.home() / "ordner_berater_undo.json"
PAPIERKORB = Path.home() / "ordner_berater_papierkorb"
PAPIERKORB.mkdir(exist_ok=True)

# Datei-Kategorien
META_ENDUNGEN = {'.docx', '.pptx', '.xlsx', '.pdf', '.jpg', '.jpeg', '.png'}
NAME_ENDUNGEN = {'.py', '.txt', '.html', '.css', '.js', '.csv', '.json',
                 '.xml', '.md', '.tex', '.r', '.ipynb'}

## 1 — State-Definition

Der `ClusterState` ist das zentrale Datenobjekt, das durch alle Nodes fließt.  
Jeder Node liest daraus und gibt ein **partielles Dict** zurück, das gemergt wird.

In [ ]:
class ClusterState(TypedDict):
    """Zentraler Zustand des Clustering-Workflows."""

    # Eingabe-Parameter
    startpfad: str
    min_cluster: int
    max_cluster: int
    anweisung: str                  # Initiale Steuer-Anweisung (optional)

    # Scan-Ergebnisse
    clustering_dateien: list        # Liste[Dict] — indizierte Dateien
    nicht_indizierte: list          # Liste[Dict] — Dateien ohne Index
    name_zu_pfad: dict              # dateiname → absoluter Pfad
    scan_info: str                  # Zusammenfassung des Scans

    # Clustering-Ergebnisse
    aktueller_vorschlag: list       # Liste[Dict] — Cluster mit Dateizuordnung
    ni_zuordnung: list              # Liste[Dict] — Zuordnung nicht-indizierter
    meldungen: list                 # Validierungs-Hinweise
    vorschlag_tabelle: str          # Formatierte Markdown-Tabelle

    # Human-in-the-Loop
    user_input: str                 # Feedback oder "__ausfuehren__"

    # Ergebnis
    bericht: str                    # Ausführungsbericht
    fehler: str                     # Fehlermeldungen

## 2 — Skills laden

Die LLM-Anweisungen liegen in `clustering_skills.md`.  
So können Prompt-Regeln und **persistente Anweisungen** ohne Code-Änderung angepasst werden.

In [ ]:
def skills_laden(pfad: Path = SKILLS_PFAD) -> dict:
    """Liest die Skills-Datei und gibt Sektionen als Dict zurück."""
    if not pfad.exists():
        print(f"⚠ Skills-Datei nicht gefunden: {pfad}")
        return {}

    text = pfad.read_text(encoding="utf-8")
    sektionen = {}
    aktuelle_sektion = None
    zeilen = []

    for line in text.splitlines():
        if line.startswith("## "):
            if aktuelle_sektion:
                sektionen[aktuelle_sektion] = "\n".join(zeilen).strip()
            aktuelle_sektion = line[3:].strip()
            zeilen = []
        else:
            zeilen.append(line)

    if aktuelle_sektion:
        sektionen[aktuelle_sektion] = "\n".join(zeilen).strip()

    return sektionen


SKILLS = skills_laden()
print("Geladene Skill-Sektionen:", list(SKILLS.keys()))

## 3 — Hilfsfunktionen

Metadaten-Extraktion, Verzeichnis-Scan, Validierung, Formatierung.  
Diese Funktionen sind **reine Logik** — kein LLM, kein State.

In [ ]:
def metadaten_lesen(dateipfad: str) -> str:
    """Liest Stichwörter aus Datei-Metadaten. Gibt String zurück."""
    ext = Path(dateipfad).suffix.lower()
    try:
        if ext == '.docx':
            from docx import Document
            doc = Document(dateipfad)
            return doc.core_properties.keywords or ""

        elif ext == '.pptx':
            from pptx import Presentation
            prs = Presentation(dateipfad)
            return prs.core_properties.keywords or ""

        elif ext == '.xlsx':
            from openpyxl import load_workbook
            wb = load_workbook(dateipfad, read_only=True)
            kw = wb.properties.keywords or ""
            wb.close()
            return kw

        elif ext == '.pdf':
            import fitz
            doc = fitz.open(dateipfad)
            meta = doc.metadata or {}
            teile = [meta.get('keywords', ''), meta.get('subject', '')]
            doc.close()
            return ", ".join(t for t in teile if t)

        elif ext in ('.jpg', '.jpeg'):
            from PIL import Image
            img = Image.open(dateipfad)
            exif = img._getexif() or {}
            desc = exif.get(270, '')
            xp_kw = exif.get(40092, b'')
            if isinstance(xp_kw, bytes):
                xp_kw = xp_kw.decode('utf-16-le', errors='ignore').rstrip('\x00')
            return ", ".join(t for t in [desc, xp_kw] if t)

        elif ext == '.png':
            from PIL import Image
            img = Image.open(dateipfad)
            info = img.info or {}
            teile = [info.get('Keywords', ''), info.get('Description', '')]
            return ", ".join(t for t in teile if t)

    except Exception as e:
        print(f"  Metadaten-Fehler bei {dateipfad}: {e}")

    return ""


def verzeichnis_scannen(startpfad: str) -> tuple:
    """Scannt rekursiv, kategorisiert Dateien, sammelt Metadaten."""
    clustering_dateien = []
    nicht_indizierte = []
    name_zu_pfad = {}
    namens_zaehler = Counter()

    for root, dirs, files in os.walk(startpfad):
        for datei in sorted(files):
            pfad = Path(root) / datei
            ext = pfad.suffix.lower()
            name_ohne_ext = pfad.stem
            quellordner = str(root)

            namens_zaehler[name_ohne_ext] += 1
            if namens_zaehler[name_ohne_ext] > 1:
                eindeutiger_name = f"{name_ohne_ext}__dup{namens_zaehler[name_ohne_ext]}"
            else:
                eindeutiger_name = name_ohne_ext

            eintrag = {
                'pfad': str(pfad),
                'name': eindeutiger_name,
                'original_name': name_ohne_ext,
                'ext': ext,
                'quellordner': quellordner,
            }

            if ext in META_ENDUNGEN:
                eintrag['stichwoerter'] = metadaten_lesen(str(pfad))
                clustering_dateien.append(eintrag)
                name_zu_pfad[eindeutiger_name] = str(pfad)
            elif ext in NAME_ENDUNGEN:
                eintrag['stichwoerter'] = ''
                clustering_dateien.append(eintrag)
                name_zu_pfad[eindeutiger_name] = str(pfad)
            else:
                nicht_indizierte.append(eintrag)

    print(f"Scan: {len(clustering_dateien)} indiziert, {len(nicht_indizierte)} nicht indiziert")
    return clustering_dateien, nicht_indizierte, name_zu_pfad


def dateiliste_formatieren(dateien: list) -> str:
    """Kompakte Dateiliste im Pipe-Format."""
    zeilen = []
    for d in dateien:
        sw = d.get('stichwoerter', '')
        zeilen.append(f"{d['name']} | {sw}" if sw else d['name'])
    return "\n".join(zeilen)


def vorschlag_validieren(ergebnis: list, bekannte_namen: set) -> list:
    """Prüft ob alle Dateien zugeordnet wurden."""
    zugeordnet = set()
    for cluster in ergebnis:
        zugeordnet.update(cluster.get('dateien', []))

    fehlend = bekannte_namen - zugeordnet
    unbekannt = zugeordnet - bekannte_namen

    meldungen = []
    if fehlend:
        meldungen.append(f"Nicht zugeordnet ({len(fehlend)}): {', '.join(sorted(fehlend)[:10])}...")
    if unbekannt:
        meldungen.append(f"Unbekannte Namen ({len(unbekannt)}): {', '.join(sorted(unbekannt)[:10])}...")
    return meldungen


def nachbarn_zuordnen(clustering_ergebnis, nicht_indizierte, clustering_dateien):
    """Ordnet nicht-indizierte Dateien per Quellordner-Nachbarschaft zu."""
    datei_zu_cluster = {}
    for cluster in clustering_ergebnis:
        for dateiname in cluster.get('dateien', []):
            datei_zu_cluster[dateiname] = cluster['ordner']

    ordner_zu_indizierte = {}
    for d in clustering_dateien:
        ordner = d['quellordner']
        if ordner not in ordner_zu_indizierte:
            ordner_zu_indizierte[ordner] = []
        ordner_zu_indizierte[ordner].append(d['name'])

    zuordnung = []
    for ni in nicht_indizierte:
        nachbarn = ordner_zu_indizierte.get(ni['quellordner'], [])

        cluster_haeufigkeit = Counter()
        for nachbar in nachbarn:
            if nachbar in datei_zu_cluster:
                cluster_haeufigkeit[datei_zu_cluster[nachbar]] += 1

        if cluster_haeufigkeit:
            ziel = cluster_haeufigkeit.most_common(1)[0][0]
            leitdatei = next(n for n in nachbarn if datei_zu_cluster.get(n) == ziel)
        else:
            ziel = '_sonstige'
            leitdatei = None

        zuordnung.append({
            'pfad': ni['pfad'],
            'name': ni['name'],
            'ext': ni.get('ext', ''),
            'ziel_cluster': ziel,
            'leitdatei': leitdatei,
        })
    return zuordnung


def vorschlag_als_tabelle(clustering_ergebnis, ni_zuordnung=None):
    """Formatiert den Vorschlag als Markdown-Tabelle."""
    zeilen = ["| Ordner | Beschreibung | Dateien |",
              "|--------|-------------|---------|"]

    for c in clustering_ergebnis:
        name = c.get('ordner', '?')
        beschr = c.get('beschreibung', '')
        anzahl = len(c.get('dateien', []))
        zeilen.append(f"| {name} | {beschr} | {anzahl} |")

    if ni_zuordnung:
        ni_pro_cluster = Counter(z['ziel_cluster'] for z in ni_zuordnung)
        if ni_pro_cluster:
            zeilen.append("")
            zeilen.append("**Nicht-indizierte Dateien (Nachbarschafts-Zuordnung):**")
            zeilen.append("")
            zeilen.append("| Ziel-Ordner | Anzahl |")
            zeilen.append("|-------------|--------|")
            for ordner, anz in ni_pro_cluster.most_common():
                zeilen.append(f"| {ordner} | {anz} |")

    return "\n".join(zeilen)

## 4 — LLM-Interface

Prompt-Bau aus Skills + LM-Studio-Aufruf.  
Die Prompts werden **dynamisch** aus `clustering_skills.md` zusammengesetzt.

In [ ]:
def prompt_erstvorschlag(dateien, min_cluster, max_cluster, anweisung=""):
    """Baut den Erstvorschlag-Prompt aus Skills + Parametern."""
    dateiliste = dateiliste_formatieren(dateien)

    regeln = SKILLS.get("ordnernamen_regeln", "Nur a-z, 0-9, Unterstriche. Max 40 Zeichen.")
    ausgabeformat = SKILLS.get("ausgabeformat",
        'Antworte als JSON-Array: [{"ordner": "...", "beschreibung": "...", "dateien": [...]}]')

    # Persistente Anweisungen aus Skills
    persistente = SKILLS.get("persistente_anweisungen", "").strip()

    # Anweisungs-Block zusammensetzen (persistent + aktuell)
    anweisungs_teile = []
    if persistente:
        anweisungs_teile.append(f"Dauerhafte Vorgaben: {persistente}")
    if anweisung.strip():
        anweisungs_teile.append(f"Zusaetzliche Anweisung: {anweisung.strip()}")

    anweisung_block = ""
    if anweisungs_teile:
        anweisung_block = "\n\n" + "\n".join(anweisungs_teile)

    return f"""Du erhaeltst eine Liste von Dateien mit optionalen Stichwoertern.
Gruppiere sie in {min_cluster} bis {max_cluster} thematische Cluster.

Regeln fuer Ordnernamen:
{regeln}

{ausgabeformat}{anweisung_block}

Dateien:
{dateiliste}"""


def prompt_korrektur(aktueller_stand, dateien, feedback):
    """Baut den Korrektur-Prompt aus Skills + aktuellem Stand."""
    dateiliste = dateiliste_formatieren(dateien)
    stand_json = json.dumps(aktueller_stand, ensure_ascii=False, indent=2)
    regeln = SKILLS.get("ordnernamen_regeln", "Nur a-z, 0-9, Unterstriche. Max 40 Zeichen.")

    return f"""Aktuelle Clusterung:
{stand_json}

Vollstaendige Dateiliste mit Stichwoertern:
{dateiliste}

Feedback: {feedback}

Ueberarbeite die Clusterung gemaess dem Feedback.
Regeln fuer Ordnernamen:
{regeln}
Antworte im selben JSON-Format. Keine Erklaerungen, nur JSON."""


def llm_anfrage(prompt: str) -> list:
    """Sendet Prompt an LM Studio, gibt geparsten JSON zurück."""
    payload = {
        "model": MODELL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATUR,
    }

    try:
        response = requests.post(API_URL, json=payload, timeout=300)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"API-Fehler: {e}")

    antwort_text = response.json()['choices'][0]['message']['content']

    # Qwen3 <think>-Tags entfernen
    antwort_text = re.sub(r'<think>.*?</think>', '', antwort_text, flags=re.DOTALL)
    # Markdown-Fences entfernen
    antwort_text = re.sub(r'^```(?:json)?\s*', '', antwort_text.strip())
    antwort_text = re.sub(r'\s*```$', '', antwort_text.strip())

    try:
        return json.loads(antwort_text)
    except json.JSONDecodeError as e:
        raise RuntimeError(f"JSON-Parsing fehlgeschlagen: {e}\n\nLLM-Antwort:\n{antwort_text[:500]}")

## 5 — Ausführung und Undo

Dateien verschieben + Undo-Log schreiben.  
`undo_letzte_aktion()` lebt **außerhalb** des Graphen — sie greift direkt auf das Log zu.

In [ ]:
def dateien_verschieben(startpfad, clustering_ergebnis, ni_zuordnung, name_zu_pfad):
    """Erstellt Ordner, verschiebt Dateien, schreibt Undo-Log."""
    startpfad = Path(startpfad)
    undo_eintraege = []
    verschoben = 0
    fehler = []

    alle_ordner = {c['ordner'] for c in clustering_ergebnis}
    if ni_zuordnung and any(z['ziel_cluster'] == '_sonstige' for z in ni_zuordnung):
        alle_ordner.add('_sonstige')

    for ordner in alle_ordner:
        (startpfad / ordner).mkdir(exist_ok=True)

    # Indizierte Dateien
    for cluster in clustering_ergebnis:
        zielordner = startpfad / cluster['ordner']
        for dateiname in cluster.get('dateien', []):
            quell_pfad = name_zu_pfad.get(dateiname)
            if not quell_pfad or not Path(quell_pfad).exists():
                fehler.append(f"Nicht gefunden: {dateiname}")
                continue

            quell = Path(quell_pfad)
            ziel = zielordner / quell.name

            if ziel.exists() and ziel != quell:
                stamm, endung = ziel.stem, ziel.suffix
                zaehler = 1
                while ziel.exists():
                    ziel = zielordner / f"{stamm}_{zaehler}{endung}"
                    zaehler += 1

            if quell == ziel:
                continue

            try:
                shutil.move(str(quell), str(ziel))
                undo_eintraege.append({'von': str(quell), 'nach': str(ziel)})
                verschoben += 1
            except Exception as e:
                fehler.append(f"Verschiebe-Fehler {quell.name}: {e}")

    # Nicht-indizierte Dateien
    if ni_zuordnung:
        for eintrag in ni_zuordnung:
            quell = Path(eintrag['pfad'])
            zielordner = startpfad / eintrag['ziel_cluster']
            zielordner.mkdir(exist_ok=True)
            ziel = zielordner / quell.name

            if ziel.exists() and ziel != quell:
                stamm, endung = ziel.stem, ziel.suffix
                zaehler = 1
                while ziel.exists():
                    ziel = zielordner / f"{stamm}_{zaehler}{endung}"
                    zaehler += 1

            if quell == ziel:
                continue

            try:
                shutil.move(str(quell), str(ziel))
                undo_eintraege.append({'von': str(quell), 'nach': str(ziel)})
                verschoben += 1
            except Exception as e:
                fehler.append(f"Verschiebe-Fehler {quell.name}: {e}")

    # Undo-Log
    undo_daten = {
        'zeitstempel': datetime.now().isoformat(),
        'startpfad': str(startpfad),
        'aktionen': undo_eintraege,
    }
    bisherige = []
    if UNDO_LOG.exists():
        try:
            bisherige = json.loads(UNDO_LOG.read_text(encoding='utf-8'))
        except:
            pass
    bisherige.append(undo_daten)
    UNDO_LOG.write_text(json.dumps(bisherige, ensure_ascii=False, indent=2), encoding='utf-8')

    # Leere Ordner aufräumen
    leere_entfernt = 0
    for root, dirs, files in os.walk(str(startpfad), topdown=False):
        if root == str(startpfad):
            continue
        if not os.listdir(root):
            os.rmdir(root)
            leere_entfernt += 1

    bericht = f"Verschoben: {verschoben} Dateien, {leere_entfernt} leere Ordner entfernt."
    if fehler:
        bericht += f"\n\nFehler ({len(fehler)}):\n" + "\n".join(fehler[:20])
    return bericht


def undo_letzte_aktion():
    """Macht die letzte Verschiebe-Aktion rückgängig (außerhalb des Graphen)."""
    if not UNDO_LOG.exists():
        return "Kein Undo-Log vorhanden."

    alle = json.loads(UNDO_LOG.read_text(encoding='utf-8'))
    if not alle:
        return "Undo-Log ist leer."

    letzte = alle.pop()
    rueckgaengig = 0
    for aktion in reversed(letzte['aktionen']):
        try:
            quell = Path(aktion['nach'])
            ziel = Path(aktion['von'])
            ziel.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(quell), str(ziel))
            rueckgaengig += 1
        except Exception as e:
            print(f"Undo-Fehler: {e}")

    UNDO_LOG.write_text(json.dumps(alle, ensure_ascii=False, indent=2), encoding='utf-8')
    return f"{rueckgaengig} Dateien zurückverschoben."

## 6 — Graph-Nodes

Jeder Node ist eine Funktion `f(state) → dict`.  
Das zurückgegebene Dict wird in den State **gemergt** (nicht ersetzt).

| Node | Aufgabe | Besonderheit |
|------|---------|-------------|
| `node_scannen` | Verzeichnis scannen | Reine I/O |
| `node_clustern` | LLM-Erstvorschlag | Nutzt `anweisung` aus State |
| `node_review` | Auf User warten | `interrupt()` — Human-in-the-Loop |
| `node_ueberarbeiten` | LLM-Korrektur | Liest `user_input` als Feedback |
| `node_ausfuehren` | Dateien verschieben | Schreibt Undo-Log |

In [ ]:
def node_scannen(state: ClusterState) -> dict:
    """Scannt das Verzeichnis und kategorisiert Dateien."""
    startpfad = state["startpfad"]

    if not startpfad or not Path(startpfad).is_dir():
        return {"fehler": f"Ungültiger Pfad: {startpfad}"}

    cd, ni, n2p = verzeichnis_scannen(startpfad)
    return {
        "clustering_dateien": cd,
        "nicht_indizierte": ni,
        "name_zu_pfad": n2p,
        "scan_info": f"Indiziert: {len(cd)}, Nicht-indiziert: {len(ni)}",
    }


def node_clustern(state: ClusterState) -> dict:
    """Ruft das LLM für den Erstvorschlag auf (mit optionaler Anweisung)."""
    cd = state["clustering_dateien"]
    if not cd:
        return {"fehler": "Keine indizierten Dateien gefunden."}

    prompt = prompt_erstvorschlag(
        cd,
        state.get("min_cluster", 5),
        state.get("max_cluster", 10),
        state.get("anweisung", ""),
    )

    try:
        ergebnis = llm_anfrage(prompt)
    except RuntimeError as e:
        return {"fehler": str(e)}

    bekannte = {d['name'] for d in cd}
    meldungen = vorschlag_validieren(ergebnis, bekannte)
    ni_z = nachbarn_zuordnen(ergebnis, state.get("nicht_indizierte", []), cd)
    tabelle = vorschlag_als_tabelle(ergebnis, ni_z)

    return {
        "aktueller_vorschlag": ergebnis,
        "ni_zuordnung": ni_z,
        "meldungen": meldungen,
        "vorschlag_tabelle": tabelle,
        "fehler": "",
    }


def node_review(state: ClusterState) -> dict:
    """Human-in-the-Loop: Pausiert den Graphen und wartet auf User-Entscheidung.

    Der User sendet entweder:
      - '__ausfuehren__'  → weiter zu node_ausfuehren
      - Feedback-Text     → weiter zu node_ueberarbeiten
    """
    user_input = interrupt({
        "tabelle": state.get("vorschlag_tabelle", ""),
        "meldungen": state.get("meldungen", []),
        "scan_info": state.get("scan_info", ""),
    })
    return {"user_input": user_input}


def route_nach_review(state: ClusterState) -> str:
    """Bedingte Kante: Entscheidet ob ausgeführt oder überarbeitet wird."""
    if state.get("user_input") == "__ausfuehren__":
        return "ausfuehren"
    return "ueberarbeiten"


def node_ueberarbeiten(state: ClusterState) -> dict:
    """Ruft das LLM mit dem User-Feedback für eine Korrektur auf."""
    feedback = state.get("user_input", "")
    cd = state["clustering_dateien"]

    prompt = prompt_korrektur(state["aktueller_vorschlag"], cd, feedback)

    try:
        ergebnis = llm_anfrage(prompt)
    except RuntimeError as e:
        return {"fehler": str(e)}

    bekannte = {d['name'] for d in cd}
    meldungen = vorschlag_validieren(ergebnis, bekannte)
    ni_z = nachbarn_zuordnen(ergebnis, state.get("nicht_indizierte", []), cd)
    tabelle = vorschlag_als_tabelle(ergebnis, ni_z)

    return {
        "aktueller_vorschlag": ergebnis,
        "ni_zuordnung": ni_z,
        "meldungen": meldungen,
        "vorschlag_tabelle": tabelle,
        "fehler": "",
    }


def node_ausfuehren(state: ClusterState) -> dict:
    """Führt die Verschiebungen aus und schreibt das Undo-Log."""
    bericht = dateien_verschieben(
        state["startpfad"],
        state["aktueller_vorschlag"],
        state.get("ni_zuordnung", []),
        state["name_zu_pfad"],
    )
    return {"bericht": bericht}

## 7 — Graph-Definition

Hier wird der `StateGraph` zusammengebaut:
1. **Nodes** registrieren
2. **Kanten** definieren (linear + bedingt)
3. **Checkpointer** anhängen (MemorySaver für Session-Persistenz)
4. **Kompilieren**

Die Überarbeitungsschleife `review → ueberarbeiten → review` ist eine **bedingte Schleife** —  
ein zentrales Pattern für agentenbasierte Workflows.

In [ ]:
# === Graph bauen ===
builder = StateGraph(ClusterState)

# Nodes registrieren
builder.add_node("scannen",       node_scannen)
builder.add_node("clustern",      node_clustern)
builder.add_node("review",        node_review)
builder.add_node("ueberarbeiten", node_ueberarbeiten)
builder.add_node("ausfuehren",    node_ausfuehren)

# Kanten: Linearer Ablauf bis zum Review-Punkt
builder.add_edge(START,       "scannen")
builder.add_edge("scannen",   "clustern")
builder.add_edge("clustern",  "review")

# Bedingte Kante: Review → Ausführen ODER Überarbeiten
builder.add_conditional_edges(
    "review",
    route_nach_review,
    {"ausfuehren": "ausfuehren", "ueberarbeiten": "ueberarbeiten"},
)

# Schleife: Überarbeiten → zurück zu Review
builder.add_edge("ueberarbeiten", "review")

# Ende
builder.add_edge("ausfuehren", END)

# Kompilieren mit Checkpointer
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Graph kompiliert. Nodes:", list(builder.nodes.keys()))

## 8 — Gradio-UI

Die UI ist ein **dünner Wrapper** über dem Graphen:
- `btn_starten` → `graph.invoke(initial_state)` — läuft bis zum `interrupt` in `review`
- `btn_ueberarbeiten` → `graph.invoke(Command(resume=feedback))` — setzt fort
- `btn_ausfuehren` → `graph.invoke(Command(resume="__ausfuehren__"))` — setzt fort bis END
- `btn_undo` → direkte Funktion (außerhalb des Graphen)

In [ ]:
# Session-Zustand für Gradio (hält nur die thread_id)
session = {"config": None}


def btn_starten(startpfad, min_c, max_c, anweisung):
    """Startet den Graphen: Scannen → Clustern → Review (Interrupt)."""
    if not startpfad or not Path(startpfad).is_dir():
        return "Ungültiger Pfad.", "", gr.update(interactive=False)

    thread_id = str(uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    session["config"] = config

    try:
        graph.invoke(
            {
                "startpfad": startpfad,
                "min_cluster": int(min_c),
                "max_cluster": int(max_c),
                "anweisung": anweisung.strip(),
            },
            config,
        )
    except Exception as e:
        return f"Fehler: {e}", "", gr.update(interactive=True)

    # State nach dem Interrupt auslesen
    state = graph.get_state(config).values
    fehler = state.get("fehler", "")
    if fehler:
        return f"**Fehler:** {fehler}", "", gr.update(interactive=True)

    scan_info = state.get("scan_info", "")
    tabelle = state.get("vorschlag_tabelle", "")
    meldungen = state.get("meldungen", [])
    if meldungen:
        tabelle += "\n\n**Hinweise:**\n" + "\n".join(f"- {m}" for m in meldungen)

    return f"**Scan:** {scan_info}", tabelle, gr.update(interactive=True)


def btn_ueberarbeiten(feedback):
    """Setzt den Graphen mit Feedback fort → Überarbeiten → Review (Interrupt)."""
    if not session.get("config"):
        return "Kein aktiver Vorschlag. Erst starten."
    if not feedback.strip():
        return "Bitte Feedback eingeben."

    try:
        graph.invoke(Command(resume=feedback.strip()), session["config"])
    except Exception as e:
        return f"Fehler: {e}"

    state = graph.get_state(session["config"]).values
    fehler = state.get("fehler", "")
    if fehler:
        return f"**Fehler:** {fehler}"

    tabelle = state.get("vorschlag_tabelle", "")
    meldungen = state.get("meldungen", [])
    if meldungen:
        tabelle += "\n\n**Hinweise:**\n" + "\n".join(f"- {m}" for m in meldungen)
    return tabelle


def btn_ausfuehren():
    """Setzt den Graphen mit '__ausfuehren__' fort → Dateien verschieben → END."""
    if not session.get("config"):
        return "Kein aktiver Vorschlag."

    try:
        graph.invoke(Command(resume="__ausfuehren__"), session["config"])
    except Exception as e:
        return f"Fehler: {e}"

    state = graph.get_state(session["config"]).values
    return state.get("bericht", "Ausführung abgeschlossen.")


def btn_undo():
    """Undo — lebt außerhalb des Graphen."""
    return undo_letzte_aktion()


# === Gradio-App ===
with gr.Blocks(title="Ordner-Clusterung (LangGraph)", theme=gr.themes.Soft()) as app:
    gr.Markdown("## Ordner-Clusterung mit LangGraph")

    with gr.Row():
        inp_pfad = gr.Textbox(label="Startpfad", placeholder="C:/Daten/Chemie/Klasse9", scale=3)
        inp_min  = gr.Number(label="Min", value=3, minimum=1, maximum=50, precision=0, scale=1)
        inp_max  = gr.Number(label="Max", value=10, minimum=1, maximum=50, precision=0, scale=1)

    inp_anweisung = gr.Textbox(
        label="Steuer-Anweisung (optional)",
        placeholder="z.B. 'Trenne Redox von Saeure-Base' oder 'Ein Ordner pro Kapitel'",
        lines=2,
    )

    with gr.Row():
        btn_start = gr.Button("Scannen & Clustern", variant="primary")

    txt_scan_info = gr.Markdown("")
    txt_vorschlag = gr.Markdown("", label="Vorschlag")

    gr.Markdown("---")
    inp_feedback = gr.Textbox(
        label="Feedback zur Überarbeitung",
        placeholder="z.B. 'Zu viele kleine Cluster' oder 'Fasse X und Y zusammen'",
        lines=2,
    )

    with gr.Row():
        btn_korrektur = gr.Button("Überarbeiten lassen", variant="secondary")
        btn_run       = gr.Button("Ausführen", variant="stop")
        btn_zurueck   = gr.Button("Undo", variant="secondary")

    txt_status = gr.Markdown("")

    # === Events ===
    btn_start.click(
        fn=btn_starten,
        inputs=[inp_pfad, inp_min, inp_max, inp_anweisung],
        outputs=[txt_scan_info, txt_vorschlag, btn_korrektur],
    )
    btn_korrektur.click(
        fn=btn_ueberarbeiten,
        inputs=[inp_feedback],
        outputs=[txt_vorschlag],
    )
    btn_run.click(
        fn=btn_ausfuehren,
        inputs=[],
        outputs=[txt_status],
    )
    btn_zurueck.click(
        fn=btn_undo,
        inputs=[],
        outputs=[txt_status],
    )

In [ ]:
app.launch(inbrowser=True)